In [2]:
from torchvision import transforms,datasets
from torch.utils.data import DataLoader
import torch.nn as nn
from pathlib import Path
import torch.optim as optim
import torchvision.models as models
import copy
import torch
import datetime
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.insert(0, project_root)
from app.DATABASE.DB_FUNC import add_model
"""
для исследования структуры модели
import torchvision.models as models

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
print(model) # Выведет полную структуру

Для разморозки/заморозки слоев
model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

# заморозим всё
for param in model.parameters():
    param.requires_grad = False

# Размораживаем только layer4 (последний свёрточный блок) и fc(full connection - классификационный слой)
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True
"""

'\nдля исследования структуры модели\nimport torchvision.models as models\n\nmodel = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)\nprint(model) # Выведет полную структуру\n\nДля разморозки/заморозки слоев\nmodel = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)\n\n# заморозим всё\nfor param in model.parameters():\n    param.requires_grad = False\n\n# Размораживаем только layer4 (последний свёрточный блок) и fc(full connection - классификационный слой)\nfor param in model.layer4.parameters():\n    param.requires_grad = True\nfor param in model.fc.parameters():\n    param.requires_grad = True\n'

In [3]:
# Для загрузки изображений
IMG_SIZE = 224
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [4]:
#формирование dataset'ов
path = Path('../data')
train_dataset = datasets.ImageFolder(
    root=path / 'training_set',
    transform=train_transform
)
val_dataset = datasets.ImageFolder(
    root=path / 'test_set',
    transform=test_transform
)
print(train_dataset.class_to_idx)

{'cats': 0, 'dogs': 1}


In [5]:
#dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size = 32,
    shuffle=True,
    num_workers=8
)
val_loader = DataLoader(
    val_dataset,
    batch_size = 32,
    shuffle=False,
    num_workers=8
)

In [6]:
#модель
model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)

In [7]:
for param in model.parameters():
    param.requires_grad = False

num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 2)

#ставим GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
#функция обучения
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    
    run_loss = 0.0
    corrects = 0
    total = 0
    
    for images, labels in loader:
        images,labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        run_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        corrects += (predicted == labels).sum().item()

    return run_loss / len(loader), 100 * corrects / total
#функция валидации
def validate(model, loader, loss_fn, device):
    model.eval()

    run_loss = 0.0
    corrects = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images,labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = loss_fn(outputs, labels)

            run_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            corrects += (predicted == labels).sum().item()

    return run_loss / len(loader), 100 * corrects / total
    

In [8]:
epochs = 10
best_acc = 0.0
best_loss = float('inf')
best_model_wts = copy.deepcopy(model.state_dict())
best_optimizer_state = copy.deepcopy(optimizer.state_dict())
best_epoch = 0
trained_model_path = f'C:/Users/pc/Desktop/project_cv/app/models/model_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}.pth'
for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, loss_fn, device)

    if val_acc > best_acc:
        best_acc = val_acc
        best_loss = val_loss
        best_epoch = epoch + 1
        best_model_wts = copy.deepcopy(model.state_dict())
        best_optimizer_state = copy.deepcopy(optimizer.state_dict())
    print(f'Epoch {epoch + 1}/{epochs}, Train loss: {train_loss:.4f}, Train acc: {train_acc:.2f}%, Val loss: {val_loss:.4f}, Val acc: {val_acc:.2f}%')

model.load_state_dict(best_model_wts)
torch.save({
    'epoch': best_epoch,
    'model_state_dict': best_model_wts,
    'optimizer_state_dict': best_optimizer_state,
    'best_acc': best_acc,
    'val_loss': best_loss,
}, trained_model_path)
add_model(trained_model_path,best_acc)
print(f"\nОбучение завершено. Лучшая точность на валидации: {best_acc:.2f}%")


Epoch 1/10, Train loss: 0.1646, Train acc: 94.23%, Val loss: 0.0876, Val acc: 97.11%
Epoch 2/10, Train loss: 0.1130, Train acc: 95.62%, Val loss: 0.0757, Val acc: 97.46%
Epoch 3/10, Train loss: 0.1041, Train acc: 95.82%, Val loss: 0.0757, Val acc: 97.31%
Epoch 4/10, Train loss: 0.0948, Train acc: 96.32%, Val loss: 0.0715, Val acc: 97.26%
Epoch 5/10, Train loss: 0.0993, Train acc: 96.17%, Val loss: 0.0667, Val acc: 97.41%
Epoch 6/10, Train loss: 0.0988, Train acc: 96.09%, Val loss: 0.0713, Val acc: 97.11%
Epoch 7/10, Train loss: 0.0933, Train acc: 96.13%, Val loss: 0.0714, Val acc: 97.26%
Epoch 8/10, Train loss: 0.0966, Train acc: 96.13%, Val loss: 0.0661, Val acc: 97.26%
Epoch 9/10, Train loss: 0.0947, Train acc: 96.12%, Val loss: 0.0677, Val acc: 97.21%
Epoch 10/10, Train loss: 0.0921, Train acc: 96.37%, Val loss: 0.0649, Val acc: 97.31%

Обучение завершено. Лучшая точность на валидации: 97.46%
